# 07. Linear SVM
This notebook trains, tunes and evaluates a **linear Support Vector Machine (SVM)** for
spoiler detection.

* only the saved **training split** is used for every decision; the **test split** is used
  once, in section 10, after everything is frozen;
* folds come from `generate_tfidf_folds`, so TF-IDF and other encoders are fitted on training
  rows only (no leakage);
* metrics: spoiler-class precision, recall, F1, macro-F1 and average precision.

## 1. Setup

**Stages and data size.** Kernel check, initial tuning and backward elimination use a
10,000-review random sample of the training split (`sample_size`).
Sections 8–9 always use every training row. Set `sample_size = None` to use full training partition.

**Class imbalance.** Only about 26% of reviews are spoilers. `imbalance = "class_weight"` makes
each mistake on a spoiler cost about 2.8× more (`class_weight="balanced"`), so the model does
not simply predict "non-spoiler" for everything.

**SVM hyperparameter grid**
* `C`: the soft-margin trade-off, searched on a log scale.
* `loss`: `"hinge"` is the standard SVM loss; `"squared_hinge"` squares it, punishing large
  violations more.
* `penalty`: `"l2"` is the usual SVM. `"l1"` sets unhelpful weights to exactly zero, so the
  model selects its own words. L1 only works with squared hinge.

**TF-IDF grid.** Unigrams vs unigrams + bigrams, `min_df=3`,
at most 5,000 terms per text field, so differences come from the model, not the features.

**Selection metric.** Mean validation **macro-F1**.
Accuracy is
recorded but never used for selection.


`max_drop = 0.005` is the elimination tolerance (section 6), fixed *before* looking at results.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Start in the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))

from src.prepare_features import INITIAL_FEATURES, prepare_features
from src.split_data import RANDOM_STATE
from src.svm import (
    tune_linear_svm, backward_elimination, refit_linear_svm, compare_kernels,
    top_features, block_importance,
)

# ---- data size and experiment switches ----
sample_size = 10000            # sections 3-7 only; sections 8-9 always use all training rows
k = 5
group_by = "review_text"       # "movie_id" for unseen movies
remove_spoilers = False        # True for spoiler-warning ablation
imbalance = "class_weight"     # "class_weight", "oversample" or "none"
selection_metric = "macro_f1"
initial_features = INITIAL_FEATURES.copy()
max_drop = 0.005
max_iter = 5000
tol = 1e-4

# ---- SVM grid (L1 + hinge is not a valid combination) ----
model_grid = [
    {"penalty": ["l2"], "loss": ["squared_hinge", "hinge"],
     "C": [0.001, 0.01, 0.03, 0.1, 0.3, 1.0]},
    {"penalty": ["l1"], "loss": ["squared_hinge"], "C": [0.01, 0.1, 1.0]},
]
full_model_grid = model_grid   

# ---- TF-IDF grid ----
tfidf_grid = {"ngram_range": [(1, 1), (1, 2)], "min_df": [3], "max_features": [5000]}

# ---- kernel check (section 3) ----
run_kernel_check = False
kernel_features = ["review_text"]          # the main signal; kernel question is about the text space
kernel_sample_sizes = (1000, 2000, 4000)   # enough to see the time curve bend
kernel_C_values = (0.1, 1.0)               # drop C=10, the slowest setting

experiment = f"linear_svm_{group_by}" + ("_ablated" if remove_spoilers else "")
print("Experiment:", experiment)

## 2. Load Training Data

Read the saved training split in chunks and join the movie details with `prepare_features`.

The sample is drawn uniformly without replacement *before* grouped CV, using the shared seed.
The test split is not opened here.

In [ ]:
split_dir = PROJECT_ROOT / "data/splits"
if group_by == "movie_id":
    split_dir = split_dir / "movie_disjoint"
train_path = split_dir / "IMDB_reviews_train.json"
movies_path = PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json"
if not train_path.exists() or not movies_path.exists():
    raise FileNotFoundError(
        "Run download_data.py, data_cleaning.py and split_data.py first "
        "(add --group-by movie_id for the movie-disjoint split)."
    )

with train_path.open(encoding="utf-8") as source:
    total_training_rows = sum(1 for _ in source)
selected_rows = None
if sample_size is not None and sample_size < total_training_rows:
    selected_rows = np.random.default_rng(RANDOM_STATE).choice(
        total_training_rows, size=sample_size, replace=False,
    )

movies = pd.read_json(movies_path, lines=True, convert_dates=False)


def load_training(rows=None, columns=None):
    # Prepared features + labels + movie IDs for the chosen rows (all rows if None).
    chunks = []
    with pd.read_json(train_path, lines=True, convert_dates=False, chunksize=10000) as reader:
        for chunk in reader:
            if rows is not None:
                chunk = chunk.loc[chunk.index.isin(rows)]
            prepared = prepare_features(chunk, movies)
            prepared["is_spoiler"] = chunk["is_spoiler"].to_numpy()
            prepared["movie_id"] = chunk["movie_id"].to_numpy()
            prepared["source_row"] = chunk.index.to_numpy()
            chunks.append(prepared if columns is None else prepared[columns])
    return pd.concat(chunks, ignore_index=True)


model_data = load_training(selected_rows)
source_row_indices = model_data.pop("source_row").to_numpy()
print(f"Using {len(model_data):,} of {total_training_rows:,} training reviews.")
print(f"Spoiler prevalence: {model_data['is_spoiler'].mean():.2%}")

## 3. Kernel Check: Linear vs RBF SVM

On random training samples of 1,000, 2,000 and 4,000 reviews, take the **first grouped CV
fold** and fit three models on identical TF-IDF matrices of **review text only**:

* `LinearSVC`: the model used in this notebook;
* `SVC(kernel="linear")`: the same kind of boundary, but solved through the kernel (dual) method;
* `SVC(kernel="rbf")`: a non-linear kernel SVM.

Each is tried at C = 0.1 and 1.0; the plot shows the best C per model and size.
Review text is the main signal, and the question is whether a curved boundary helps in the
TF-IDF space. Kernel SVMs were too slow to compare on all eight feature blocks (an earlier
attempt ran for over 30 minutes without finishing).

This uses training data only and is a *design check*; its scores are not reported as results.
Set `run_kernel_check = False` to skip it.

**Results**

| Sample size | LinearSVC | SVC (linear kernel) | SVC (RBF) | Fit time: LinearSVC vs RBF |
|---|---|---|---|---|
| 1,000 | 0.565 | 0.558 | 0.554 | 0.10 s vs 4.3 s |
| 2,000 | 0.633 | 0.614 | 0.593 | 0.27 s vs 31.6 s |
| 4,000 | 0.610 | 0.605 | 0.580 | 0.84 s vs 63.6 s |

(validation macro-F1 at the best C)

1. **No macro-F1 gain.** The RBF SVM did not detect spoilers any better than LinearSVC.
2. **Higher cost.** The kernel cost grows about with n². When Extrapolated to the 459,000 training reviews, it will take too long to train and predict.
3. **Almost every review is a support vector.** The RBF SVM kept 94% of training reviews as
   support vectors, this shows how much the two classes overlap.

**Why a linear SVM is sufficient**

TF-IDF already represents each review with thousands of word features, and our kernel check showed that a non-linear (RBF) boundary did not improve macro-F1 over a linear one. A linear SVM is also far cheaper to train at our data size and gives one weight per word, letting us see which words drive spoiler predictions.


In [ ]:
if run_kernel_check:
    kernel_tfidf = {name: values[0] for name, values in tfidf_grid.items()}
    kernel_results = compare_kernels(
        model_data, feature_columns=initial_features, sample_sizes=kernel_sample_sizes,
        C_values=kernel_C_values, tfidf_params=kernel_tfidf, k=k, group_by=group_by,
        random_state=RANDOM_STATE, imbalance=imbalance,
    )
    best_per_model = (kernel_results.sort_values("macro_f1", ascending=False)
                      .drop_duplicates(["sample_size", "model"])
                      .sort_values(["model", "sample_size"]))
    best_per_model["support_vector_share"] = (best_per_model["support_vectors"]
                                              / best_per_model["train_rows"])
    display(best_per_model[["model", "sample_size", "C", "macro_f1", "f1", "average_precision",
                            "fit_seconds", "predict_seconds", "support_vector_share"]])

    fig_kernel, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for name, rows in best_per_model.groupby("model"):
        axes[0].plot(rows["sample_size"], rows["macro_f1"], marker="o", label=name)
        axes[1].plot(rows["sample_size"], rows["fit_seconds"], marker="o", label=name)
    axes[0].set(xlabel="Training sample size", ylabel="Validation macro-F1 (best C)",
                title="Does a kernel improve accuracy?")
    axes[1].set(xlabel="Training sample size", ylabel="Fit time (seconds, log scale)",
                yscale="log", title="How does training time grow?")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend()
    fig_kernel.tight_layout()
    plt.show()

    # Rough extrapolation of fit time to the full training split (time ~ n^slope).
    extrapolation = []
    for name, rows in best_per_model.groupby("model"):
        if len(rows) >= 2 and (rows["fit_seconds"] > 0).all():
            slope, intercept = np.polyfit(np.log(rows["sample_size"]), np.log(rows["fit_seconds"]), 1)
            extrapolation.append({"model": name, "growth_exponent": slope,
                                  "estimated_full_fit_hours": np.exp(intercept)
                                  * total_training_rows ** slope / 3600})
    display(pd.DataFrame(extrapolation))

## 4. Initial Tuning With All Features

Every candidate (SVM settings × TF-IDF settings) uses the same grouped folds. The TF-IDF vocabulary is built from each
fold's training reviews only (so validation reviews never leak in) and shared by all SVM
settings for that fold. The candidate with the highest mean validation macro-F1 is selected.

Reported per candidate (means/std over the 5 **validation** folds):
* `macro_f1`, `f1`, `precision`, `recall`, `average_precision`: validation scores at threshold 0;
* `train_macro_f1`: the same score on the fold's training rows. A large gap between training and
  validation means overfitting;
* `margin_width`, `inside_margin`, `nonzero_weights`: SVM diagnostics (explained in section 5);
* `converged`: whether the solver finished within `max_iter` in every fold. Unconverged
  candidates stay visible but cannot be selected.

In [ ]:
baseline_summary, baseline_fold_scores, baseline_config = tune_linear_svm(
    model_data, model_grid=model_grid, tfidf_grid=tfidf_grid,
    feature_columns=initial_features, k=k, group_by=group_by, random_state=RANDOM_STATE,
    imbalance=imbalance, remove_spoilers=remove_spoilers, selection_metric=selection_metric,
    max_iter=max_iter, tol=tol,
)
if baseline_config is None:
    raise RuntimeError("No candidate converged in every fold. Increase max_iter and rerun.")
show_columns = ["penalty", "loss", "C", "ngram_range", "macro_f1_mean", "macro_f1_std",
                "f1_mean", "precision_mean", "recall_mean", "average_precision_mean",
                "train_macro_f1_mean", "inside_margin", "nonzero_weights", "seconds", "converged"]
display(baseline_summary[show_columns])
print("Best all-feature configuration:", baseline_config)

## 5. Inspect C: Bias–Variance and the Margin

Hold the best loss, penalty and TF-IDF setting fixed and look at how the regularisation strength C changes the model.

**Left chart: training vs validation macro-F1**
* Small C: both scores low and close together → **underfitting** (margin too wide, too simple).
* Large C: training score keeps rising while validation flattens or drops → **overfitting**.
* The best C sits where validation peaks. If that is at the edge of the grid, extend the grid.

**Right chart: margin**
* *Margin width* $2/\lVert w\rVert$ shrinks as C grows: the model accepts a narrower street
  to fit more training reviews.
* *Share of training rows inside the margin* ($y\,f(x) < 1$): these rows produce hinge loss,
  and for a hinge-loss SVM they are exactly the **support vectors**. If most rows are inside the
  margin even at the best C, the two classes overlap heavily in this feature space. That is
  expected for spoilers, where many reviews are ambiguous, and explains why F1 is far from 1.

In [ ]:
best_model = baseline_config["model"]
same_setting = baseline_summary[
    (baseline_summary["penalty"] == best_model["penalty"])
    & (baseline_summary["loss"] == best_model["loss"])
    & baseline_summary["ngram_range"].map(lambda value: value == baseline_config["tfidf"]["ngram_range"])
].sort_values("C")

fig_c, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for column, label in [("train_macro_f1", "Training"), ("macro_f1", "Validation")]:
    axes[0].errorbar(same_setting["C"], same_setting[f"{column}_mean"],
                     yerr=same_setting[f"{column}_std"], marker="o", capsize=3, label=label)
axes[0].axvline(best_model["C"], color="grey", linestyle=":", label=f"Chosen C = {best_model['C']}")
axes[0].set(xscale="log", xlabel="C (log scale)", ylabel="Macro-F1",
            title=f"{best_model['penalty'].upper()} / {best_model['loss']}: training vs validation")
axes[0].legend()
axes[1].plot(same_setting["C"], same_setting["inside_margin"], marker="o", color="tab:purple",
             label="Share of training rows inside margin")
axes[1].set(xscale="log", xlabel="C (log scale)", ylabel="Share of training rows", ylim=(0, 1.05))
twin = axes[1].twinx()
twin.plot(same_setting["C"], same_setting["margin_width"], marker="s", color="tab:orange",
          label="Margin width 2/||w||")
twin.set(ylabel="Margin width", yscale="log")
axes[1].set_title("Margin vs C")
lines = axes[1].get_legend_handles_labels()[0] + twin.get_legend_handles_labels()[0]
axes[1].legend(lines, [line.get_label() for line in lines], loc="center right")
for ax in axes:
    ax.grid(alpha=0.25)
fig_c.tight_layout()
plt.show()

# Hinge vs squared hinge, and L1 vs L2, at their own best C.
display(baseline_summary.sort_values("macro_f1_mean", ascending=False)
        .drop_duplicates(["penalty", "loss", "ngram_range"])
        [["penalty", "loss", "ngram_range", "C", "macro_f1_mean", "f1_mean", "nonzero_weights"]])

## 6. Backward Feature Elimination

Freeze the SVM and TF-IDF settings from section 4. At each step, try removing each remaining
block once, keep the removal with the best mean validation score, and continue down to one
block. Then choose the **smallest** subset whose score is within `max_drop` (0.5 percentage
points) of the best score on the whole path. The tolerance is measured from the best score, not
per step, so small losses can't accumulate. 

This is a greedy search: it does not test every
possible subset. 
Folds stay identical at every step.

In [ ]:
selected_features, elimination_path, elimination_trials = backward_elimination(
    model_data, baseline_config, feature_columns=initial_features, max_drop=max_drop,
    selection_metric=selection_metric, k=k, group_by=group_by, random_state=RANDOM_STATE,
    imbalance=imbalance, remove_spoilers=remove_spoilers, max_iter=max_iter, tol=tol,
)
print("Selected features:", selected_features)
print("Removed features:", [f for f in initial_features if f not in selected_features])
display(elimination_path)

## 7. Elimination Plot

The line follows the best removal at each step. The shaded band is ±1 fold standard deviation
(not a confidence interval). The dashed line is the allowed floor; the red point is the
smallest acceptable subset.

In [ ]:
fig_elim, ax = plt.subplots(figsize=(12, 5.5))
steps = elimination_path["step"].to_numpy()
means = elimination_path["score_mean"].to_numpy()
stds = elimination_path["score_std"].to_numpy()
chosen = elimination_path.loc[elimination_path["selected"]].iloc[0]
ax.plot(steps, means, marker="o", label="Greedy elimination path")
ax.fill_between(steps, means - stds, means + stds, alpha=0.15, label="±1 fold std")
ax.axhline(means.max() - max_drop, color="tab:green", linestyle="--", label="Allowed floor")
ax.scatter(chosen["step"], chosen["score_mean"], color="tab:red", s=130, zorder=3,
           label="Selected subset")
ax.set_xticks(steps, [
    f"{row.n_features} blocks\n" + ("all" if row.step == 0 else f"- {row.removed_feature}")
    for row in elimination_path.itertuples()
], rotation=30, ha="right")
ax.set(ylabel=f"Mean validation {selection_metric}", title="Linear SVM: backward feature elimination")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig_elim.tight_layout()
plt.show()

## 8. Retune the Selected Features on the Full Training Split

Load **every** training row (still not the test split), keeping only the selected inputs plus
the columns needed for grouping and labels. Rerun the grid with grouped CV. The folds differ
from the sample's, so the comparison table states each stage's data scope; the score changes
are not a paired estimate of the elimination's effect.

In [ ]:
if len(model_data) == total_training_rows:
    full_model_data = model_data
else:
    keep = list(dict.fromkeys(selected_features + ["review_text", "movie_id", "is_spoiler"]))
    full_model_data = load_training(columns=keep)
print(f"Full-training retuning: {len(full_model_data):,} reviews | features: {selected_features}")

summary, fold_scores, best_config = tune_linear_svm(
    full_model_data, model_grid=full_model_grid, tfidf_grid=tfidf_grid,
    feature_columns=selected_features, k=k, group_by=group_by, random_state=RANDOM_STATE,
    imbalance=imbalance, remove_spoilers=remove_spoilers, selection_metric=selection_metric,
    max_iter=max_iter, tol=tol,
)
if best_config is None:
    raise RuntimeError("No retuned candidate converged in every fold. Increase max_iter and rerun.")
display(summary[[c for c in show_columns if c in summary]])
print("Final configuration:", best_config)

first = baseline_summary[baseline_summary["converged"]].iloc[0]
last = summary[summary["converged"]].iloc[0]
picked = elimination_path.loc[elimination_path["selected"]].iloc[0]
comparison = pd.DataFrame([
    {"stage": "All features, tuned", "n_blocks": len(initial_features), "data": "sample",
     "rows": len(model_data), "score_mean": first[f"{selection_metric}_mean"],
     "score_std": first[f"{selection_metric}_std"]},
    {"stage": "Selected features, fixed settings", "n_blocks": len(selected_features),
     "data": "sample", "rows": len(model_data), "score_mean": picked["score_mean"],
     "score_std": picked["score_std"]},
    {"stage": "Selected features, retuned", "n_blocks": len(selected_features),
     "data": "full training", "rows": len(full_model_data),
     "score_mean": last[f"{selection_metric}_mean"], "score_std": last[f"{selection_metric}_std"]},
])
display(comparison)

## 9. Refit, Save and Interpret the Final Model

Fit fresh encoders and SVM weights on every training review, using the selected features
(section 6) and the final settings (section 8). CV only chose the *settings*; the fold models are
discarded, because each fold had its own vocabulary.

**Reading the weights.** Positive weights push a review towards "spoiler", negative weights
towards "non-spoiler". The block-level table sums the absolute weights per input, giving a rough
idea of how much the model relies on each input. 

In [ ]:
model, fit_info = refit_linear_svm(
    full_model_data, best_config, feature_columns=selected_features, random_state=RANDOM_STATE,
    imbalance=imbalance, remove_spoilers=remove_spoilers, max_iter=max_iter, tol=tol,
)
print("Final fit:", {key: round(value, 4) if isinstance(value, float) else value
                     for key, value in fit_info.items()})

output_dir = PROJECT_ROOT / "data/models" / f"{experiment}_full_backward"
output_dir.mkdir(parents=True, exist_ok=True)
baseline_summary.to_csv(output_dir / "baseline_cv_summary.csv")
baseline_fold_scores.to_csv(output_dir / "baseline_cv_fold_scores.csv", index=False)
elimination_path.to_json(output_dir / "elimination_path.json", orient="records", indent=2)
elimination_trials.to_json(output_dir / "elimination_trials.json", orient="records", indent=2)
summary.to_csv(output_dir / "cv_summary.csv")
fold_scores.to_csv(output_dir / "cv_fold_scores.csv", index=False)
fig_c.savefig(output_dir / "c_curve_and_margin.png", dpi=150, bbox_inches="tight")
fig_elim.savefig(output_dir / "backward_elimination.png", dpi=150, bbox_inches="tight")
if run_kernel_check:
    kernel_results.to_csv(output_dir / "kernel_check.csv", index=False)
    fig_kernel.savefig(output_dir / "kernel_check.png", dpi=150, bbox_inches="tight")
np.save(output_dir / "training_row_indices.npy", np.arange(len(full_model_data)))
np.save(output_dir / "selection_row_indices.npy", source_row_indices)
joblib.dump({
    "pipeline": model, "best_config": best_config, "feature_columns": selected_features,
    "fit_info": fit_info, "training_path": str(train_path),
    "training_rows": len(full_model_data), "feature_selection_rows": len(model_data),
    "total_training_rows": total_training_rows, "k": k, "group_by": group_by,
    "remove_spoilers": remove_spoilers, "imbalance": imbalance,
    "selection_metric": selection_metric, "decision_threshold": 0.0,
    "initial_features": initial_features, "baseline_config": baseline_config,
    "max_drop": max_drop, "model_grid": model_grid, "full_model_grid": full_model_grid,
    "tfidf_grid": tfidf_grid, "max_iter": max_iter, "tol": tol,
    "random_state": RANDOM_STATE, "sklearn_version": sklearn.__version__,
}, output_dir / "model.joblib", compress=3)
print("Saved model and CV results:", output_dir)

weights = top_features(model, n=20)
weights.to_csv(output_dir / "top_features.csv", index=False)
display(block_importance(model))
fig_w, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (direction, rows), color in zip(axes, weights.groupby("direction", sort=False),
                                        ["tab:red", "tab:blue"]):
    rows = rows.iloc[::-1]
    ax.barh(rows["feature"], rows["weight"], color=color)
    ax.set(title=f"Top {direction} features", xlabel="SVM weight")
fig_w.tight_layout()
fig_w.savefig(output_dir / "top_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Held-Out Test Evaluation (run once)

Run only after the features, settings and threshold are frozen. This section reloads the saved
pipeline and the matching test split and processes it in chunks. Nothing is fitted or tuned here. 

A review is predicted as a spoiler when its SVM decision value is **> 0** (the side of the
boundary it falls on). Average precision uses the decision values as ranking scores.
Test scores must **not** be used to change anything above.

**What is reported**
* **Test metrics:** macro-F1 (primary metric), spoiler-class precision, recall and F1, average
  precision (how well the decision values rank spoilers above non-spoilers), and accuracy (for
  completeness only, since always predicting "non-spoiler" already scores about 0.74).
* **Confusion matrix:** counts of correct and incorrect predictions for each class.
* **Precision-recall curve:** precision vs recall as the threshold moves. The dotted line is the
  spoiler rate, which is the average precision of a random guess.
* **Decision-value histogram:** where spoiler and non-spoiler test reviews fall relative to the
  boundary (0) and the margin (−1 and +1).

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
from src.clean_spoilers import clean_text
from src.metrics import evaluate

evaluation_dir = PROJECT_ROOT / "data/models" / f"{experiment}_full_backward"
saved = joblib.load(evaluation_dir / "model.joblib")
test_model = saved["pipeline"]
test_path = Path(saved["training_path"]).with_name("IMDB_reviews_test.json")
test_movies = pd.read_json(PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json",
                           lines=True, convert_dates=False)

prediction_chunks = []
with pd.read_json(test_path, lines=True, convert_dates=False, chunksize=10000) as reader:
    for test_reviews in reader:
        test_features = prepare_features(test_reviews, test_movies)[saved["feature_columns"]]
        decision = test_model.decision_function(test_features)
        prediction_chunks.append(pd.DataFrame({
            "test_row": test_reviews.index.to_numpy(),
            "movie_id": test_reviews["movie_id"].to_numpy(),
            "is_spoiler": test_reviews["is_spoiler"].to_numpy().astype(int),
            "predicted_spoiler": (decision > saved["decision_threshold"]).astype(int),
            "decision_value": decision,
            "has_warning_phrase": [clean_text(t) != t for t in test_reviews["review_text"]],
            "review_excerpt": test_reviews["review_text"].str.slice(0, 300).to_numpy(),
        }))
test_predictions = pd.concat(prediction_chunks, ignore_index=True)
test_scores = evaluate(test_predictions["is_spoiler"], test_predictions["predicted_spoiler"],
                       test_predictions["decision_value"])
test_scores["average_precision"] = test_scores.pop("pr_auc")
metric_order = ["macro_f1", "precision", "recall", "f1", "average_precision", "accuracy"]
test_metrics = pd.DataFrame([test_scores])
test_confusion = pd.DataFrame(
    [[test_scores["tn"], test_scores["fp"]], [test_scores["fn"], test_scores["tp"]]],
    index=["Actual non-spoiler", "Actual spoiler"],
    columns=["Predicted non-spoiler", "Predicted spoiler"],
)
print(f"Held-out test reviews: {len(test_predictions):,}")
display(test_metrics[metric_order].T.rename(columns={0: "Test score"}))
display(test_confusion)

fig_t, axes = plt.subplots(1, 2, figsize=(13, 4.5))
PrecisionRecallDisplay.from_predictions(test_predictions["is_spoiler"],
                                        test_predictions["decision_value"], ax=axes[0],
                                        name="Linear SVM")
axes[0].axhline(test_predictions["is_spoiler"].mean(), color="grey", linestyle=":",
                label="Random-guess baseline (AP = spoiler rate)")
axes[0].legend()
axes[0].set_title("Precision-recall curve (test)")
for label, name in [(0, "Non-spoiler"), (1, "Spoiler")]:
    axes[1].hist(test_predictions.loc[test_predictions["is_spoiler"] == label, "decision_value"],
                 bins=80, alpha=0.55, density=True, label=name)
axes[1].axvline(0, color="black", linestyle="--", label="Decision boundary (0)")
for edge in (-1, 1):
    axes[1].axvline(edge, color="grey", linestyle=":")
axes[1].set(xlabel="SVM decision value  w·x + b", ylabel="Density",
            title="Where test reviews fall (dotted lines = margin)")
axes[1].legend()
fig_t.tight_layout()
plt.show()

test_predictions.to_csv(evaluation_dir / "test_predictions.csv", index=False)
test_metrics.to_csv(evaluation_dir / "test_metrics.csv", index=False)
test_confusion.to_csv(evaluation_dir / "test_confusion_matrix.csv")
fig_t.savefig(evaluation_dir / "test_pr_and_margin.png", dpi=150, bbox_inches="tight")

results_row = {"model": experiment, "split": "shared_text_grouped",
               **{m: round(float(test_scores[m]), 4)
                  for m in ["accuracy", "precision", "recall", "f1", "macro_f1"]},
               **{c: int(test_scores[c]) for c in ("tn", "fp", "fn", "tp")},
               "pr_auc": round(float(test_scores["average_precision"]), 4),
               "spoiler_rate_test": round(float(test_predictions["is_spoiler"].mean()), 4),
               "features": "+".join(saved["feature_columns"]),
               **{key: saved["best_config"]["model"][key] for key in ("penalty", "loss", "C")},
               "ngram_range": str(saved["best_config"]["tfidf"]["ngram_range"])}
pd.DataFrame([results_row]).to_csv(evaluation_dir / "test_results_row.csv", index=False)
print("Saved test predictions and metrics:", evaluation_dir)